<a href="https://colab.research.google.com/github/GustavoABrandao/Projeto---IA-Facens/blob/main/Conhe%C3%A7a_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("PySpark Docker Example") \
    .getOrCreate()

In [5]:
dataframe = spark.read.csv("/content/Predição_de_preços_ML.csv", header=True, inferSchema=True)
num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 310797


In [13]:
dataframe.show(100)

+----------+--------------------+--------------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+---------+---------+-----------+--------------------+--------------------+------+-----+---------+----------+--------------------+
|        id|                 url|              region|          region_url|price|year|manufacturer|               model|condition|  cylinders|  fuel|odometer|title_status|transmission|              VIN|drive|     size|     type|paint_color|           image_url|         description|county|state|      lat|      long|        posting_date|
+----------+--------------------+--------------------+--------------------+-----+----+------------+--------------------+---------+-----------+------+--------+------------+------------+-----------------+-----+---------+---------+-----------+--------------------+--------------------+------+-----+---------+----------+--------

In [43]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr


from pyspark.sql.functions import col, regexp_extract

# =========================
# 1. DROP E LIMPEZA ROBUSTA
# =========================
# PRIMEIRO: Limpamos extraindo apenas os números e convertendo
df = df.withColumn("price", regexp_extract(col("price"), r"(\d+)", 0).cast("double"))
df = df.withColumn("year", regexp_extract(col("year"), r"(\d{4})", 0).cast("double"))
df = df.withColumn("odometer", regexp_extract(col("odometer"), r"(\d+)", 0).cast("double"))

df = df.withColumn("lat", col("lat").cast("float"))
df = df.withColumn("long", col("long").cast("float"))

# SEGUNDO: Dropamos os nulos *depois* da conversão, para garantir que o lixo que virou nulo suma
df = df.dropna(subset=["price", "year", "odometer"])

df = df.drop(
    "id", "url", "region_url", "VIN",
    "image_url", "description", "county",
    "posting_date", "size"
)

# =========================
# 2. FILTROS E OUTLIERS
# =========================

# Regras físicas (hard rules) - Colocamos tetos máximos de sanidade para evitar overflow
df = df.filter(
    (col("price") > 100) & (col("price") < 10000000) & # Teto de 10 milhões para evitar bilhões
    (col("year") > 1900) & (col("year") <= 2025) &
    (col("odometer") < 5000000) & # Teto de 5 milhões no odômetro
    (col("lat").between(-90.0, 90.0)) # Usando .0 para forçar comparação em Float/Double
)

# Remoção de extremos estatísticos usando um erro relativo saudável (0.01)
for c in ["price", "odometer"]:
    # O valor 0.01 no final permite 1% de margem de erro no quantil, evitando travamentos
    q_low, q_high = df.approxQuantile(c, [0.01, 0.99], 0.01)
    df = df.filter((col(c) >= q_low) & (col(c) <= q_high))
# =========================
# 3. Colunas categóricas
# =========================

# One-hot (nominais)
categorical_cols = [
    "region", "manufacturer", "fuel", "transmission",
    "drive", "type", "paint_color", "state", "title_status"
]

# StringIndexer (ordinal ou alta cardinalidade)
index_cols = ["condition", "model"]

# =========================
# 4. Criar stages
# =========================
stages = []

# Index + OneHot para categóricas nominais
for col_name in categorical_cols:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )

    encoder = OneHotEncoder(
        inputCol=col_name + "_index",
        outputCol=col_name + "_onehot",
        dropLast=True
    )

    stages += [indexer, encoder]

# Apenas index para ordinal/alta cardinalidade
for col_name in index_cols:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )

    stages.append(indexer)


# =========================
# 5. Pipeline
# =========================
pipeline = Pipeline(stages=stages)
model = pipeline.fit(df)
df_prepared = model.transform(df)
df_prepared.createOrReplaceTempView("cars_prepared")


ArithmeticException: [CAST_OVERFLOW] The value 3.0095488E9 of the type "FLOAT" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [39]:
features_sql = spark.sql("""
SELECT *
FROM cars_prepared
LIMIT 20
""")

features_sql.show()


+--------------------+-----+----+------------+-----+---------+---------+----+--------+------------+------------+-----+----+-----------+-----+----+----+-----------+--------------+----------+------------+-----------------+------------------+-------------------+----------+-----------+------------------+-------------------+-----------+------------+----------+-----------+-----------------+------------------+-----------+---------------+------------------+-------------------+---------------+-----------+
|              region|price|year|manufacturer|model|condition|cylinders|fuel|odometer|title_status|transmission|drive|type|paint_color|state| lat|long|price_clean|odometer_clean|year_clean|region_index|    region_onehot|manufacturer_index|manufacturer_onehot|fuel_index|fuel_onehot|transmission_index|transmission_onehot|drive_index|drive_onehot|type_index|type_onehot|paint_color_index|paint_color_onehot|state_index|   state_onehot|title_status_index|title_status_onehot|condition_index|model_ind